In [1]:
import sys
import os
import site
import importlib

print("Ege Üniversitesi scraper'ı için çalışma ortamı hazırlanıyor...")

# Kütüphaneler önceki adımda kurulduğu için sadece yolları tanıtıyoruz
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)
    
importlib.invalidate_caches()

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
import time

print("✅ TEST BAŞARILI: Ortam hazır, kütüphaneler bağlandı!")

Ege Üniversitesi scraper'ı için çalışma ortamı hazırlanıyor...
✅ TEST BAŞARILI: Ortam hazır, kütüphaneler bağlandı!


In [3]:
import requests
import time
import pandas as pd
import os
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

OUTPUT_FILE = "data/03_ege_university_abstracts.csv"
TARGET_LIMIT = 20000 

if not os.path.isfile(OUTPUT_FILE):
    pd.DataFrame(columns=["source", "title", "abstract", "subjects"]).to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

# GÜNCELLENEN DSpace 7 OAI Endpoint'i (Eski /oai/request rotası /server/oai/request olarak değişti)
base_url = "https://acikerisim.ege.edu.tr/server/oai/request"
params = {'verb': 'ListRecords', 'metadataPrefix': 'oai_dc'}

# 403 Forbidden Yememek İçin Zırh
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

medical_keywords = [
    'tıp', 'sağlık', 'hastane', 'klinik', 'tedavi', 'hasta', 
    'cerrahi', 'sendrom', 'enfeksiyon', 'hekim', 'hemşire', 
    'farmakoloji', 'psikiyatri', 'pediatri', 'kardiyoloji', 'diyabet',
    'metabolizma', 'endokrin'
]

dataset = []
count = 0
retry_count = 0 # Sonsuz döngü kırıcı

print("Ege Üniversitesi DSpace 7 veri hattı başlatılıyor...")

with tqdm(total=TARGET_LIMIT, desc="Ege Üni. Tezleri") as pbar:
    while count < TARGET_LIMIT:
        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=30)
            
            if response.status_code == 429:
                print("\n[Uyarı] Rate Limit (429). 60 sn bekleniyor...")
                time.sleep(60)
                continue
            elif response.status_code != 200:
                # SESSİZ DÖNGÜYÜ KIRAN VE DURUMU EKRANA BASAN KISIM
                print(f"\n[Hata] Sunucu {response.status_code} kodu döndürdü. Yanıt: {response.text[:150]}")
                retry_count += 1
                if retry_count > 3:
                    print("Üst üste çok fazla sunucu hatası alındı. İşlem durduruluyor, endpoint'i kontrol etmeliyiz.")
                    break
                time.sleep(15)
                continue
            
            # Başarılı İstek: Hata sayacını sıfırla
            retry_count = 0
            
            soup = BeautifulSoup(response.content, 'xml')
            
            error = soup.find('error')
            if error:
                print(f"\nOAI-PMH Hatası veya Veritabanı Sonu: {error.text}")
                break

            records = soup.find_all('record')
            
            for record in records:
                header = record.find('header')
                if header and header.get('status') == 'deleted':
                    continue
                    
                metadata = record.find('metadata')
                if metadata:
                    title_elem = metadata.find('dc:title')
                    desc_elem = metadata.find('dc:description')
                    subject_elems = metadata.find_all('dc:subject')
                    
                    if title_elem and desc_elem:
                        title = title_elem.text.strip()
                        description = desc_elem.text.strip()
                        subjects = ", ".join([sub.text for sub in subject_elems])
                        
                        text_to_search = (title + " " + description + " " + subjects).lower()
                        
                        if any(word in text_to_search for word in medical_keywords):
                            dataset.append({
                                'source': 'Ege_University_Thesis',
                                'title': title,
                                'abstract': description,
                                'subjects': subjects
                            })
                            count += 1
                            pbar.update(1)
                            
                            # Olası RAM şişmelerini ve kopmaları önlemek için 50'de bir yaz
                            if len(dataset) >= 50:
                                pd.DataFrame(dataset).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")
                                dataset = []
                                
                            if count >= TARGET_LIMIT:
                                break
            
            token = soup.find('resumptionToken')
            if token and token.text:
                params = {'verb': 'ListRecords', 'resumptionToken': token.text}
                time.sleep(2) 
            else:
                print("\nTüm Ege Üniversitesi veritabanı tarandı.")
                break

        except requests.exceptions.RequestException as e:
            print(f"\n[Bağlantı Hatası] {e} - 20 sn bekleniyor...")
            time.sleep(20)
            continue
        except Exception as e:
            print(f"\nBeklenmeyen hata: {e}")
            break

    # Kalan verileri diske dök
    if dataset:
        pd.DataFrame(dataset).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")

print(f"\nEge Üniversitesi işlemi tamamlandı! Veriler '{OUTPUT_FILE}' dosyasına eklendi.")

Ege Üniversitesi DSpace 7 veri hattı başlatılıyor...


Ege Üni. Tezleri:   0%|          | 0/20000 [00:00<?, ?it/s]


Ege Üniversitesi işlemi tamamlandı! Veriler 'data/03_ege_university_abstracts.csv' dosyasına eklendi.
